In [1]:
import torch

assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun this cell.')
print('GPU :', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

GPU : Tesla T4
torch: 2.11.0+cu130


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/ecg-paper'
WORK = '/content/work'

!mkdir -p {WORK}/src {WORK}/artifacts
!cp {DRIVE}/src/*.py {WORK}/src/
!cp {DRIVE}/artifacts/ds1.npz {WORK}/artifacts/

%cd {WORK}
!ls -la src/ artifacts/

Mounted at /content/drive
/content/work
artifacts/:
total 12068
drwxr-xr-x 2 root root     4096 Oct  6 16:19 .
drwxr-xr-x 4 root root     4096 Oct  6 16:19 ..
-rw------- 1 root root 12346552 Oct  6 16:19 ds1.npz

src/:
total 52
drwxr-xr-x 2 root root 4096 Oct  6 16:19 .
drwxr-xr-x 4 root root 4096 Oct  6 16:19 ..
-rw------- 1 root root 4852 Oct  6 16:19 config.py
-rw------- 1 root root 5294 Oct  6 16:19 imbalance.py
-rw------- 1 root root 7440 Oct  6 16:19 models.py
-rw------- 1 root root 9685 Oct  6 16:19 run_grid.py
-rw------- 1 root root 4382 Oct  6 16:19 train_cnn.py


In [3]:
# torch and sklearn ship with Colab; these two do not.
!pip install -q xgboost imbalanced-learn

In [4]:
import sys
import numpy as np

sys.path.insert(0, 'src')
import config as C

d = np.load('artifacts/ds1.npz')
y, pid = d['y'], d['pid']

print(f'beats    : {len(y):,}   (expect 50,977)')
print(f'patients : {len(np.unique(pid))}       (expect 22)')
print('classes  :', {c: int((y == i).sum()) for i, c in enumerate(C.CLASS_NAMES)})
print('eval on  :', C.EVAL_CLASSES, '(Q excluded)')

assert len(y) == 50977, 'ds1.npz does not match Part A — re-download from Drive'
print('\nOK')

beats    : 50,977   (expect 50,977)
patients : 22       (expect 22)
classes  : {'N': 45824, 'S': 943, 'V': 3788, 'F': 414, 'Q': 8}
eval on  : ['N', 'S', 'V', 'F'] (Q excluded)

OK


In [5]:
!python src/run_grid.py --models cnn --strategies none --rr 1 --subset 5000 --epochs 5

DS1: 5,000 beats from 22 patients
grid: 1 models x 1 strategies x 1 rr x 5 folds

/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
cnn  none         rr   macroF1 0.4431 +/- 0.0456   S=0.116 V=0.773 F=0.000   [16s]

Wrote 1 configurations to artifacts/grid_summary.json
Per-fold predictions in artifacts/preds/ — Stages 3-5 read these.


In [6]:
!rm -rf artifacts/preds artifacts/grid_summary.json
print('smoke-test output cleared')

smoke-test output cleared


In [7]:
!python src/run_grid.py --models cnn

DS1: 50,977 beats from 22 patients
grid: 1 models x 4 strategies x 2 rr x 5 folds

cnn  none         rr   macroF1 0.4920 +/- 0.0567   S=0.216 V=0.798 F=0.003   [239s]
cnn  none         norr  macroF1 0.4301 +/- 0.0275   S=0.029 V=0.720 F=0.009   [240s]
cnn  class_weight rr   macroF1 0.4971 +/- 0.0250   S=0.453 V=0.744 F=0.041   [230s]
cnn  class_weight norr  macroF1 0.3796 +/- 0.0323   S=0.041 V=0.641 F=0.000   [219s]
cnn  focal        rr   macroF1 0.5342 +/- 0.0600   S=0.503 V=0.767 F=0.159   [294s]
cnn  focal        norr  macroF1 0.3723 +/- 0.0524   S=0.062 V=0.623 F=0.037   [336s]
cnn  adasyn       rr   macroF1 0.4752 +/- 0.0550   S=0.181 V=0.719 F=0.000   [532s]
cnn  adasyn       norr  macroF1 0.4143 +/- 0.0269   S=0.024 V=0.643 F=0.089   [425s]

Wrote 8 configurations to artifacts/grid_summary.json
Per-fold predictions in artifacts/preds/ — Stages 3-5 read these.


In [8]:
!mkdir -p {DRIVE}/artifacts/preds
!cp -r {WORK}/artifacts/preds/* {DRIVE}/artifacts/preds/ 2>/dev/null
!ls {WORK}/artifacts/preds | wc -l
print('^ folds saved to Drive (40 when complete)')

40
^ folds saved to Drive (40 when complete)


In [9]:
import json

with open('artifacts/grid_summary.json') as f:
    rows = json.load(f)

for r in rows:
    pc = r['per_class_f1']
    print(f"{r['model']:4} {r['strategy']:12} rr={int(r['use_rr'])}  "
          f"macroF1 {r['macro_f1_mean']:.4f} +/- {r['macro_f1_std']:.4f}   "
          f"S={pc['S']:.3f} V={pc['V']:.3f} F={pc['F']:.3f}")

cnn  none         rr=1  macroF1 0.4920 +/- 0.0567   S=0.216 V=0.798 F=0.003
cnn  none         rr=0  macroF1 0.4301 +/- 0.0275   S=0.029 V=0.720 F=0.009
cnn  class_weight rr=1  macroF1 0.4971 +/- 0.0250   S=0.453 V=0.744 F=0.041
cnn  class_weight rr=0  macroF1 0.3796 +/- 0.0323   S=0.041 V=0.641 F=0.000
cnn  focal        rr=1  macroF1 0.5342 +/- 0.0600   S=0.503 V=0.767 F=0.159
cnn  focal        rr=0  macroF1 0.3723 +/- 0.0524   S=0.062 V=0.623 F=0.037
cnn  adasyn       rr=1  macroF1 0.4752 +/- 0.0550   S=0.181 V=0.719 F=0.000
cnn  adasyn       rr=0  macroF1 0.4143 +/- 0.0269   S=0.024 V=0.643 F=0.089
